# Credit Card Fraud Detection - K-Nearest Neighbor(KNN)

## Importing the Dependencies

Non-parametric instance-based classification using scaled distance metrics.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pylab as plt
%matplotlib inline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Read the CSV with fallback
try:
    credit_card_data = pd.read_csv("../creditcard.csv")
    print("✅ Loaded ../creditcard.csv")
except FileNotFoundError:
    credit_card_data = pd.read_csv("creditcard.csv")
    print("✅ Loaded creditcard.csv")

In [ ]:
# first 5 rows of the dataset
credit_card_data.head()

In [ ]:
# Preprocessing and scaling
scaler = StandardScaler()
df_knn = credit_card_data.copy()
df_knn['Amount_Scaled'] = scaler.fit_transform(df_knn[['Amount']])
df_knn.drop(columns=['Time', 'Amount'], inplace=True)

X = df_knn.drop(columns=['Class'])
y = df_knn['Class']

# Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Full Train: {X_train.shape[0]:,} rows, Test: {X_test.shape[0]:,} rows")

## Stratified Subsampling for Efficient KNN Execution

KNN has $O(N \times D)$ query complexity. Training on 227,000 samples would take excessive computation per test point. We evaluate KNN using an effective stratified undersample preserving 100% of fraud cases.

In [ ]:
# Create balanced training subset for KNN
train_df = pd.concat([X_train, y_train], axis=1)
fraud_train = train_df[train_df['Class'] == 1]
normal_train = train_df[train_df['Class'] == 0].sample(n=len(fraud_train) * 4, random_state=42)

subsample_train = pd.concat([fraud_train, normal_train]).sample(frac=1, random_state=42)

X_train_sub = subsample_train.drop(columns=['Class'])
y_train_sub = subsample_train['Class']

print(f"KNN Training Samples: {X_train_sub.shape[0]} ({y_train_sub.sum()} Frauds, {(y_train_sub == 0).sum()} Normal)")

## Training KNN Classifier (k=5)

In [ ]:
# Train KNN model
knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
knn.fit(X_train_sub, y_train_sub)

y_pred_knn = knn.predict(X_test)
y_prob_knn = knn.predict_proba(X_test)[:, 1]

print("=== KNN (k=5) Evaluation on Full Test Set ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_knn):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_knn):.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_knn)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', cbar=False)
plt.title(f'KNN Confusion Matrix (k=5, Recall: {recall_score(y_test, y_pred_knn):.2%})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.yticks([0.5, 1.5], ['Normal', 'Fraud'])
plt.show()

## Evaluating Different k Values

In [ ]:
# Test k values from 3 to 9 on a test sample
test_sample = pd.concat([X_test, y_test], axis=1).sample(n=5000, random_state=42)
X_test_eval = test_sample.drop(columns=['Class'])
y_test_eval = test_sample['Class']

k_values = [3, 5, 7, 9]
k_f1s = []

for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train_sub, y_train_sub)
    score = f1_score(y_test_eval, model.predict(X_test_eval), zero_division=0)
    k_f1s.append(score)
    print(f"k={k}: F1-Score = {score:.4f}")

plt.figure(figsize=(7, 4))
plt.plot(k_values, k_f1s, marker='o', color='#6f42c1', lw=2)
plt.title('KNN F1-Score vs Number of Neighbors (k)', fontsize=12, fontweight='bold')
plt.xlabel('k (Neighbors)')
plt.ylabel('F1-Score')
plt.grid(True, alpha=0.3)
plt.show()